## Local notebook setup

This notebook is intended to be run **locally** from the project root (so relative paths like `data/processed/...` work).

Data locations used in this notebook:
- Raw inputs (optional here): `data/raw/`
- Processed joined dataset (required): `data/processed/hotels_with_cities-2.parquet`


In [ ]:
# Ensure imports work when running locally.
# Run this notebook from the repo root (the folder that contains `src/` and `data/`).

import sys
from pathlib import Path

PROJECT = Path.cwd()
sys.path.insert(0, str(PROJECT))

import src
print("Imported src from:", Path(src.__file__).resolve())

## Verify local files exist

This step confirms the processed dataset exists locally before we load it.

Raw inputs under `data/raw/` are optional here (only needed if you want to regenerate processed outputs).

In [ ]:
from pathlib import Path

PROCESSED = Path("data/processed/hotels_with_cities-2.parquet")
print("processed exists:", PROCESSED.is_file(), PROCESSED)

RAW_HOTELS = Path("data/raw/hotels.csv")
RAW_WORLD = Path("data/raw/worldcitiespop.csv")
print("raw hotels exists:", RAW_HOTELS.is_file(), RAW_HOTELS)
print("raw world cities exists:", RAW_WORLD.is_file(), RAW_WORLD)

In [ ]:
# Install project requirements (local)
# If you already have dependencies installed, you can skip this.

!python3 -m pip install -q -r requirements.txt

## Data wrangling (already completed)

The data-wrangling pipeline (cleaning + standardization + feature creation + Hotels × World Cities join) has already been run, and its output is saved locally as:
- `data/processed/hotels_with_cities-2.parquet`

That processed dataset is what we will use for the rest of this notebook.

If you ever need to regenerate processed outputs from the raw CSVs, you can run the pipeline command shown next—but it is **not needed** for the normal workflow.

In [ ]:
# Do NOT run for the normal workflow.
# The processed dataset is already available at `data/processed/hotels_with_cities-2.parquet`.

RUN_CLEANING = False

if RUN_CLEANING:
    !python3 scripts/pipeline/run_cleaning.py \
        --output-dir data/processed \
        --format parquet \
        --chunksize 50000 \
        --progress-every 50000
else:
    print("Skipping cleaning pipeline (processed dataset already exists).")

## Use the processed dataset

Now we use the processed dataset created above (`data/processed/hotels_with_cities-2.parquet`) and summarize what it contains.

**This cell outputs:**
- Dataset **size** (rows, columns)
- A **5-row preview** (what one hotel record looks like)
- The **top 12 columns by % missing** (coverage snapshot)
- The **distribution of hotel star ratings** (counts per `hotel_star_rating`)

In [ ]:
from pathlib import Path
import pandas as pd

joined = Path("data/processed/hotels_with_cities-2.parquet")
if not joined.is_file():
    raise FileNotFoundError(
        f"Expected processed Parquet at {joined}. "
        "If you saved it under a different name/location, update the path here."
    )

df = pd.read_parquet(joined)

# Normalize any alias column names to the canonical schema.
try:
    from src.modeling.feature_matrix import normalize_engineered_column_names

    df = normalize_engineered_column_names(df)
except Exception:
    pass

print("processed Parquet:", joined)
print("rows:", len(df), "cols:", df.shape[1])

display(df.head(5))

na_rate = (df.isna().mean() * 100).sort_values(ascending=False)
display(na_rate.head(12).to_frame(name="% missing"))

if "hotel_star_rating" not in df.columns:
    raise KeyError("Expected column 'hotel_star_rating' not found in processed dataset.")

display(df["hotel_star_rating"].value_counts(dropna=False).sort_index().to_frame(name="count"))

## Modeling (local)

This section trains and compares multiple regression models for hotel rating prediction.

Supported models:
- `linear`, `ridge`, `lasso`, `rf`, `xgb`

Metrics reported:
- RMSE, MAE, R²
- hit rates: `|error| <= 0.5` and `|error| <= 1.0`


In [ ]:
from pathlib import Path

PROJECT_ROOT = Path(".").resolve()
OUT_DIR = Path("outputs/model_artifacts_local")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Use a sample for speed while iterating. Set to None for full dataset.
SAMPLE_ROWS = 100_000
RANDOM_STATE = 42

print("project root:", PROJECT_ROOT)
print("artifacts dir:", OUT_DIR)
print("sample rows:", SAMPLE_ROWS)


In [ ]:
# Train two baseline models (fast examples).
# This writes metrics.json and model artifacts under OUT_DIR.

sample_arg = [] if SAMPLE_ROWS is None else ["--sample", str(SAMPLE_ROWS)]

!python3 scripts/modeling/train_baseline_model.py --project-root "{PROJECT_ROOT}" --model linear --out-dir "{OUT_DIR / 'linear'}" --random-state {RANDOM_STATE} {*sample_arg}
!python3 scripts/modeling/train_baseline_model.py --project-root "{PROJECT_ROOT}" --model rf --out-dir "{OUT_DIR / 'rf'}" --random-state {RANDOM_STATE} {*sample_arg}


In [ ]:
# Run a small comparison set and summarize metrics.

import json
import subprocess
import pandas as pd

RUNS_DIR = OUT_DIR / "runs"
RUNS_DIR.mkdir(parents=True, exist_ok=True)

models_to_run = [
    ("linear", []),
    ("ridge", ["--ridge-alpha", "10"]),
    ("lasso", ["--lasso-alpha", "0.001"]),
    ("rf", ["--rf-estimators", "200", "--rf-max-depth", "20"]),
]

for model_name, extra_args in models_to_run:
    out_dir = RUNS_DIR / model_name
    out_dir.mkdir(parents=True, exist_ok=True)
    cmd = [
        "python3",
        "scripts/modeling/train_baseline_model.py",
        "--project-root",
        str(PROJECT_ROOT),
        "--model",
        model_name,
        "--out-dir",
        str(out_dir),
        "--random-state",
        str(RANDOM_STATE),
    ]
    if SAMPLE_ROWS is not None:
        cmd.extend(["--sample", str(SAMPLE_ROWS)])
    cmd.extend(extra_args)

    print("Running:", " ".join(cmd))
    subprocess.run(cmd, cwd=PROJECT_ROOT, check=True)

summary_rows = []
for model_name, _ in models_to_run:
    metrics_path = RUNS_DIR / model_name / "metrics.json"
    if not metrics_path.exists():
        continue
    metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
    summary_rows.append(
        {
            "model": model_name,
            "rows": metrics["rows"],
            "rmse": metrics["rmse"],
            "mae": metrics["mae"],
            "r2": metrics["r2"],
            "within_0_5": metrics["within_0_5"],
            "within_1_0": metrics["within_1_0"],
            "tuned": metrics.get("tuned", False),
        }
    )

results_df = pd.DataFrame(summary_rows).sort_values(by="rmse", ascending=True)
results_df

## Optional: hypothesis testing (permutation-based)

This runs a simulation-based test to check whether observed relationships for selected features are stronger than chance (correlation evidence, not causation).


In [ ]:
import subprocess
from pathlib import Path

hypo_out = OUT_DIR / "hypothesis_tests.json"
cmd = [
    "python3",
    "scripts/modeling/permutation_hypothesis_tests.py",
    "--project-root",
    str(PROJECT_ROOT),
    "--out",
    str(hypo_out),
    "--n-permutations",
    "200",
    "--random-state",
    str(RANDOM_STATE),
]
if SAMPLE_ROWS is not None:
    cmd.extend(["--sample", str(SAMPLE_ROWS)])

print("Running:", " ".join(cmd))
subprocess.run(cmd, cwd=PROJECT_ROOT, check=True)
print("Wrote:", hypo_out)


## Optional: XGBoost (local note)

On macOS, XGBoost often requires the OpenMP runtime (**OpenMP `libomp`**).

If you see an error about `libomp.dylib` missing, install it and reinstall XGBoost:

- `brew install libomp`
- restart your terminal/kernel
- `python3 -m pip install -U xgboost`

If XGBoost is not available locally, you can still run `linear`, `ridge`, `lasso`, and `rf` here, and run `xgb` in Kaggle.


In [ ]:
# Optional local XGBoost run (guarded).

RUN_XGB = False

if RUN_XGB:
    try:
        import xgboost  # noqa: F401

        print("xgboost available")
    except Exception as e:
        raise RuntimeError(
            "xgboost is not available locally. On macOS, install OpenMP runtime (libomp) then reinstall xgboost."
        ) from e

    import subprocess

    out_dir = RUNS_DIR / "xgb"
    out_dir.mkdir(parents=True, exist_ok=True)

    cmd = [
        "python3",
        "scripts/modeling/train_baseline_model.py",
        "--project-root",
        str(PROJECT_ROOT),
        "--model",
        "xgb",
        "--out-dir",
        str(out_dir),
        "--random-state",
        str(RANDOM_STATE),
        "--xgb-estimators",
        "300",
        "--xgb-max-depth",
        "8",
        "--xgb-learning-rate",
        "0.05",
    ]
    if SAMPLE_ROWS is not None:
        cmd.extend(["--sample", str(SAMPLE_ROWS)])

    print("Running:", " ".join(cmd))
    subprocess.run(cmd, cwd=PROJECT_ROOT, check=True)
else:
    print("RUN_XGB is False — skipping XGBoost locally.")
